In [1]:
from pyspark.sql import SparkSession

In [3]:
# Instantiate spark session
spark = SparkSession.builder \
    .appName("Walmart SQL Analysis") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()


In [4]:
# Load CSV as DataFrame
df = spark.read.csv("walmart_transformed_7_31_25.csv", header=True, inferSchema=True)

# Show the first few rows
df.show()


+-----+----+-----+---+------------+------------+-----------+----------+-----------+------------+-------------+--------------------+-----------+
|Month|Year|Store|Day|Weekly_Sales|Holiday_Flag|Temperature|Fuel_Price|        CPI|Unemployment|FormattedDate|FormattedHolidayFlag|HolidayType|
+-----+----+-----+---+------------+------------+-----------+----------+-----------+------------+-------------+--------------------+-----------+
|    4|2011|   33|  1|   232769.09|           0|      71.41|     3.772|128.7199355|       8.687|     1-Apr-11|         Not Holiday|Non-Holiday|
|    4|2011|   17|  1|   795859.23|           0|      39.38|     3.487|128.7199355|       6.774|     1-Apr-11|         Not Holiday|Non-Holiday|
|    4|2011|   13|  1|  1864238.64|           0|      42.49|     3.487|128.7199355|       7.193|     1-Apr-11|         Not Holiday|Non-Holiday|
|    4|2011|   32|  1|  1051121.02|           0|      44.83|     3.461|192.2691707|       8.595|     1-Apr-11|         Not Holiday|Non-H

In [5]:
# Register csv as a table
df.createOrReplaceTempView("sales_data")


In [10]:
top_5_store_sales = spark.sql("""
    SELECT Store, SUM(Weekly_Sales) as total_sales
    FROM sales_data
    GROUP BY store
    ORDER BY total_sales DESC
    LIMIT 5
""")

top_5_store_sales.show()

+-----+--------------------+
|Store|         total_sales|
+-----+--------------------+
|   20|      3.0139779246E8|
|    4|      2.9954395338E8|
|   14| 2.889999113399999E8|
|   13|2.8651770379999995E8|
|    2|      2.7538244098E8|
+-----+--------------------+



Here we can see the top 5 most productive stores over our time frame. We can see store 20 produced about $300 million in total sales from 2010-2012.

In [12]:
bottom_5_store_sales = spark.sql("""
    SELECT Store, SUM(Weekly_Sales) as total_sales
    FROM sales_data
    GROUP BY store
    ORDER BY total_sales ASC
    LIMIT 5
""")

bottom_5_store_sales.show()

+-----+--------------------+
|Store|         total_sales|
+-----+--------------------+
|   33| 3.716022195999999E7|
|   44|       4.329308784E7|
|    5| 4.547568890000001E7|
|   36|5.3412214969999984E7|
|   38|5.5159626419999994E7|
+-----+--------------------+



Here are the total sales for the bottom 5 stores. Store 33 only produced about $37 million in sales.

In [14]:
total_sales = spark.sql("""
    SELECT SUM(Weekly_Sales) AS total_sales
    FROM sales_data
""")

total_sales.show()

+-------------------+
|        total_sales|
+-------------------+
|6.737218987109989E9|
+-------------------+



Here are the total sales over the time period. From 2010-2012, all stores produced a total of about $6.73 billion in sales.